<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/advanced/06_advanced_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced LLM Evaluation Frameworks

> **Track:** AI Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
Evaluating LLMs is fundamentally different from traditional software testing. Outputs are probabilistic, subjective, and context-dependent. This notebook covers a **systematic approach** to LLM evaluation: building custom metrics, implementing LLM-as-judge, tracking regression, and generating evaluation reports.

### What You'll Learn
- Why LLM evaluation is hard
- RAGAS-style metrics (faithfulness, relevance, context precision)
- LLM-as-judge implementation
- Building an evaluation dataset
- Statistical significance in evals
- Automated regression testing for prompts

```bash
pip install openai pandas numpy scipy
```

In [ ]:
# Setup: imports and configuration
import json
import os
import time
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
from scipy import stats

import openai

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", "YOUR_KEY"))

@dataclass
class EvalSample:
    """A single evaluation sample for a RAG system."""
    question: str
    ground_truth: str
    retrieved_context: list[str]
    generated_answer: str = ""

# Sample evaluation dataset
EVAL_DATASET: list[EvalSample] = [
    EvalSample(
        question="What is the capital of France?",
        ground_truth="Paris is the capital of France.",
        retrieved_context=["France is a country in Western Europe. Its capital city is Paris."],
        generated_answer="The capital of France is Paris."
    ),
    EvalSample(
        question="Who invented the telephone?",
        ground_truth="Alexander Graham Bell is credited with inventing the telephone in 1876.",
        retrieved_context=["The telephone was invented in the 19th century. Multiple inventors were involved."],
        generated_answer="Thomas Edison invented the telephone."
    ),
    EvalSample(
        question="What is photosynthesis?",
        ground_truth="Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose.",
        retrieved_context=["Photosynthesis occurs in plant chloroplasts. It converts light energy into chemical energy stored as sugar."],
        generated_answer="Photosynthesis is how plants make food using sunlight, water, and carbon dioxide."
    ),
]

print(f"Evaluation dataset: {len(EVAL_DATASET)} samples loaded.")

## 1. LLM-as-Judge Pattern

The most flexible evaluation approach: use a powerful LLM (GPT-4, Claude) to score another LLM's outputs. The judge model is given a **structured rubric** and returns a score with reasoning.

In [ ]:
# Implement LLM-as-judge for correctness scoring

JUDGE_PROMPT = """
You are an expert evaluator. Score the following answer on a scale of 1-5.

Question: {question}
Ground Truth: {ground_truth}
Generated Answer: {generated_answer}

Scoring rubric:
5 - Completely correct, fully answers the question
4 - Mostly correct, minor omissions
3 - Partially correct, significant gaps
2 - Mostly incorrect, small correct elements
1 - Completely wrong or hallucinated

Respond with JSON: {{"score": int, "reasoning": str}}
"""

@dataclass
class JudgeResult:
    score: int
    reasoning: str
    sample_question: str

def llm_judge_correctness(sample: EvalSample) -> JudgeResult:
    """Score a generated answer for correctness using an LLM judge."""
    prompt = JUDGE_PROMPT.format(
        question=sample.question,
        ground_truth=sample.ground_truth,
        generated_answer=sample.generated_answer
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    return JudgeResult(score=result["score"], reasoning=result["reasoning"], sample_question=sample.question)

# Mock results for demo
MOCK_JUDGE_RESULTS = [
    JudgeResult(score=5, reasoning="Answer is completely correct.", sample_question=EVAL_DATASET[0].question),
    JudgeResult(score=1, reasoning="Wrong inventor named. Bell, not Edison.", sample_question=EVAL_DATASET[1].question),
    JudgeResult(score=4, reasoning="Mostly correct but missing detail about glucose.", sample_question=EVAL_DATASET[2].question),
]

for r in MOCK_JUDGE_RESULTS:
    print(f"Q: {r.sample_question[:40]}...")
    print(f"   Score: {r.score}/5 | {r.reasoning}\n")

## 2. RAG-Specific Metrics

RAG systems require specialized metrics that evaluate the retrieval + generation pipeline:
- **Faithfulness**: Is the answer grounded in the retrieved context (no hallucination)?
- **Context Relevance**: Is the retrieved context actually relevant to the question?
- **Answer Relevance**: Does the answer actually address the question?

In [ ]:
# Implement RAGAS-inspired metrics

def score_faithfulness(sample: EvalSample) -> float:
    """Score 0-1: How well is the answer supported by the retrieved context?"""
    prompt = f"""
    Context: {' '.join(sample.retrieved_context)}
    Answer: {sample.generated_answer}
    
    Is the answer fully supported by the context above? 
    Return JSON: {{"score": float between 0 and 1, "unsupported_claims": [str]}}
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["score"]

def score_context_relevance(sample: EvalSample) -> float:
    """Score 0-1: How relevant is the retrieved context to the question?"""
    prompt = f"""
    Question: {sample.question}
    Context: {' '.join(sample.retrieved_context)}
    
    How relevant is this context for answering the question?
    Return JSON: {{"score": float between 0 and 1}}
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["score"]

# Mock metric scores
mock_metrics = pd.DataFrame([
    {"question": s.question[:40], "faithfulness": f, "context_relevance": c, "correctness": r.score / 5}
    for s, f, c, r in zip(
        EVAL_DATASET,
        [1.0, 0.1, 0.9],   # faithfulness (sample 2 hallucinates Edison)
        [0.95, 0.5, 0.9],  # context relevance
        MOCK_JUDGE_RESULTS
    )
])

print(mock_metrics.to_string(index=False))
print(f"\nMean scores:")
print(mock_metrics[["faithfulness", "context_relevance", "correctness"]].mean())

## 3. Regression Testing for Prompts

When you update a prompt, you need to verify it doesn't regress on existing test cases. We implement a **prompt regression framework** that compares scores before and after a prompt change.

In [ ]:
# Regression testing framework for prompt changes

@dataclass
class EvalRun:
    """Stores results of one evaluation run."""
    run_id: str
    prompt_version: str
    scores: list[float] = field(default_factory=list)

    @property
    def mean_score(self) -> float:
        return float(np.mean(self.scores)) if self.scores else 0.0

    @property
    def std_score(self) -> float:
        return float(np.std(self.scores)) if self.scores else 0.0

def compare_prompt_versions(
    baseline: EvalRun,
    candidate: EvalRun,
    alpha: float = 0.05
) -> dict[str, Any]:
    """Statistical comparison of two prompt versions using paired t-test."""
    delta = candidate.mean_score - baseline.mean_score
    t_stat, p_value = stats.ttest_rel(candidate.scores, baseline.scores)
    significant = p_value < alpha
    direction = "IMPROVED" if delta > 0 else "REGRESSED"
    verdict = f"{direction} (p={p_value:.3f}, {'significant' if significant else 'not significant'})"

    return {
        "baseline_mean": round(baseline.mean_score, 3),
        "candidate_mean": round(candidate.mean_score, 3),
        "delta": round(delta, 3),
        "p_value": round(p_value, 3),
        "statistically_significant": significant,
        "verdict": verdict
    }

# Simulate two prompt versions
baseline_run = EvalRun(run_id="run_001", prompt_version="v1.0", scores=[0.6, 0.5, 0.7, 0.8, 0.65])
candidate_run = EvalRun(run_id="run_002", prompt_version="v1.1", scores=[0.75, 0.65, 0.85, 0.9, 0.8])

comparison = compare_prompt_versions(baseline_run, candidate_run)
print("Prompt Regression Test Results:")
for k, v in comparison.items():
    print(f"  {k}: {v}")

## 4. Evaluation Report Generation

A complete evaluation pipeline generates a structured **report** that can be tracked over time, shared with the team, and used to make deployment decisions.

In [ ]:
# Generate a comprehensive evaluation report

def generate_eval_report(
    run: EvalRun,
    metrics_df: pd.DataFrame,
    threshold: float = 0.7
) -> dict[str, Any]:
    """Build a structured evaluation report for a model run."""
    passing = metrics_df[metrics_df["correctness"] >= threshold]
    failing = metrics_df[metrics_df["correctness"] < threshold]

    report = {
        "run_id": run.run_id,
        "prompt_version": run.prompt_version,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "summary": {
            "total_samples": len(metrics_df),
            "passing": len(passing),
            "failing": len(failing),
            "pass_rate": round(len(passing) / len(metrics_df), 2),
        },
        "metric_means": {
            col: round(float(metrics_df[col].mean()), 3)
            for col in ["faithfulness", "context_relevance", "correctness"]
        },
        "failing_cases": failing["question"].tolist(),
        "recommendation": "DEPLOY" if len(passing) / len(metrics_df) >= 0.8 else "DO NOT DEPLOY"
    }
    return report

report = generate_eval_report(baseline_run, mock_metrics)
print("=== EVALUATION REPORT ===")
print(json.dumps(report, indent=2))

## Exercises

1. **Add a hallucination detector**: Implement a `detect_hallucinations(sample: EvalSample) -> list[str]` function that asks the LLM to identify specific claims in the generated answer that are NOT supported by the retrieved context. Test it on the telephone example.

2. **Build a golden dataset**: Create 10 question-answer-context triplets for a domain of your choice (cooking, history, science). Run the full evaluation pipeline on them and identify which question types score lowest.

3. **Multi-model comparison**: Evaluate the same set of 10 questions using two different models (e.g., `gpt-4o-mini` vs `gpt-4o`). Use the `compare_prompt_versions` function (treating each model as a "version") and determine if the score difference is statistically significant.